# 03 — Customer Segmentation (RFM Analysis)

**Goal:** Segment customers using the RFM model (Recency, Frequency, Monetary).
Identify VIP, Loyal, At-Risk, New, and One-Time buyers.

---

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from analysis import set_style, build_rfm, plot_rfm_segments, PALETTE

set_style()
print('Ready ✅')

In [ ]:
master = pd.read_csv('../data/processed/master.csv', parse_dates=['order_purchase_timestamp'])
print(f'Loaded {master.shape[0]:,} rows')

## 1. Build RFM Table

In [ ]:
rfm = build_rfm(master)
print(f'Customers scored: {rfm.shape[0]:,}')
rfm.head(10)

## 2. RFM Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col, label, color in zip(
    axes,
    ['recency', 'frequency', 'monetary'],
    ['Recency (days since last order)', 'Frequency (# orders)', 'Monetary (R$ total spend)'],
    [PALETTE['primary'], PALETTE['secondary'], PALETTE['accent']]
):
    ax.hist(rfm[col], bins=40, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel(col.capitalize())
    ax.set_ylabel('Count')

plt.suptitle('RFM Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Segment Summary

In [ ]:
seg_summary = (
    rfm.groupby('segment')
    .agg(
        customers=('customer_unique_id', 'count'),
        avg_recency=('recency', 'mean'),
        avg_frequency=('frequency', 'mean'),
        avg_monetary=('monetary', 'mean'),
        total_revenue=('monetary', 'sum'),
    )
    .reset_index()
    .sort_values('total_revenue', ascending=False)
)

seg_summary.style.format({
    'avg_recency': '{:.0f} days',
    'avg_frequency': '{:.1f}',
    'avg_monetary': 'R${:,.2f}',
    'total_revenue': 'R${:,.2f}',
})

## 4. Segment Charts

In [ ]:
plot_rfm_segments(rfm, save_path='../images/rfm_segments.png')

## 5. RFM Heatmap (R vs M by Segment)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    rfm['recency'], rfm['monetary'],
    c=rfm['rfm_score'], cmap='RdYlGn',
    alpha=0.4, s=15
)
plt.colorbar(scatter, label='RFM Score')
ax.set_xlabel('Recency (days)')
ax.set_ylabel('Monetary Value (R$)')
ax.set_title('Recency vs Monetary (coloured by RFM Score)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/rfm_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save RFM Table

In [ ]:
rfm.to_csv('../data/processed/rfm.csv', index=False)
print('✅ RFM table saved to data/processed/rfm.csv')